# 01 · PTQ on Qwen3-0.6B: FP16 · RTN · GPTQ · AWQ · NF4 + T4 speed
**Runtime:** one T4 GPU (Colab or Kaggle). **Writes:** `results_01_ptq.jsonl`. **Rough time:** ~3–4 h (AWQ dominates).

| config_id | What |
|---|---|
| `fp16` | Baseline + KL-divergence reference + T4 speed |
| `rtn_w4g128`, `rtn_w3g128` | Plain rounding (symmetric, group 128) |
| `gptq_w4g128`, `gptq_w3g128` | GPTQ, 128 × 2,048 tokens of C4 |
| `awq_w4g128`, `awq_w3g128` | AWQ (asymmetric), same calibration, **run in fp32** (fp16 gives NaN on a T4) |
| `gptq_w4g128_code` | GPTQ calibrated on code (calibration-shift test) |
| `nf4_dq` | bitsandbytes NF4 + double quant (no calibration) |

Eval per row: WikiText-2 ppl (full test) · code ppl · KL divergence + top-1 agreement vs FP16 · arc_easy + hellaswag (1,000 each).
Speed: vLLM, batch 1, 512 in / 128 out, median of 5 (FP16, GPTQ-4, AWQ-4).

In [ ]:
!pip install -q -U llmcompressor==0.15.0 lm-eval==0.4.13 datasets accelerate bitsandbytes   # pip warnings about gradio/cudf/bigframes are safe to ignore

## Setup
Set `PLATFORM` (`"colab"` or `"kaggle"`). The helper `edgelayer_bench.py` is fetched from the repo
(or used directly if you cloned the repo). Every finished config is saved to `RESULTS_FILE` right away,
so after a disconnect you just re-run all cells: finished configs are skipped.

In [ ]:
PLATFORM = "kaggle"                      # "colab" | "kaggle"
RESULTS_FILE = "results_01_ptq.jsonl"
REPO_RAW = "https://raw.githubusercontent.com/<your-username>/edgelayer-quantization/main"   # <- replace once

import os, sys, json, urllib.request
BENCH_DIR = {"colab": "/content/drive/MyDrive/edgelayer-bench",
             "kaggle": "/kaggle/working/edgelayer-bench"}[PLATFORM]
if PLATFORM == "colab":
    from google.colab import drive; drive.mount("/content/drive")
os.makedirs(BENCH_DIR, exist_ok=True)

# Use the repo's helper if this notebook runs from a clone, else download it once
for d in ["..", "."]:
    if os.path.exists(f"{d}/edgelayer_bench.py"):
        sys.path.insert(0, os.path.abspath(d)); break
else:
    urllib.request.urlretrieve(f"{REPO_RAW}/edgelayer_bench.py", f"{BENCH_DIR}/edgelayer_bench.py")
    sys.path.insert(0, BENCH_DIR)

In [ ]:
import torch
import edgelayer_bench as eb
rd = eb.setup(BENCH_DIR, platform=PLATFORM, results_file=RESULTS_FILE)
MODEL_ID = eb.CFG["model_id"]
eb.ensure_meta()                            # parameter counts, used for bits/weight

def done_ids():
    """config_ids that already have a row with this field (default: any row)."""
    p = rd / RESULTS_FILE
    return {json.loads(l)["config_id"] for l in open(p) if l.strip()} if p.exists() else set()

print(MODEL_ID, "|", eb.ram_gb()); eb.env_info()

In [ ]:
from transformers import AutoModelForCausalLM, BitsAndBytesConfig
from llmcompressor import oneshot
from llmcompressor.modifiers.quantization import QuantizationModifier, GPTQModifier
from llmcompressor.modifiers.awq import AWQModifier
from llmcompressor.utils import dispatch_for_generation

## FP16 baseline (also the KL-divergence reference, kept on the GPU)

In [ ]:
eb.CFG["lm_eval_batch"] = "auto"
ref, tok = eb.load_fp16()
if "fp16" not in done_ids():
    res = eb.evaluate(ref, tok, ref=None)
    eb.log_result("fp16", post=0, method="FP16", bits=16, size_gb=round(eb.get_meta()["n_params"] * 2 / 1e9, 3),
                  bpw=16.0, kld=0.0, top1_agree=100.0, **res)

## Quantize + evaluate (each row saved as soon as it finishes; finished rows are skipped)

In [ ]:
CALIB = {"c4": eb.calib_dataset(tok, "c4"), "code": eb.calib_dataset(tok, "domain")}

CONFIGS = [
    # config_id            algo    bits calib   post
    ("rtn_w4g128",         "rtn",  4,   None,   3),
    ("rtn_w3g128",         "rtn",  3,   None,   3),
    ("gptq_w4g128",        "gptq", 4,   "c4",   3),
    ("gptq_w3g128",        "gptq", 3,   "c4",   3),
    ("awq_w4g128",         "awq",  4,   "c4",   4),
    ("awq_w3g128",         "awq",  3,   "c4",   4),
    ("gptq_w4g128_code",   "gptq", 4,   "code", 4),
    # ("awq_w4g128_code",  "awq",  4,   "code", 4),   # ran out of memory on a free T4 in our runs
]
KEEP_FOR_SPEED = {"gptq_w4g128", "awq_w4g128"}

def make_recipe(algo, bits):
    if algo == "rtn":
        return QuantizationModifier(config_groups=eb.w_scheme(bits, 128, symmetric=True), ignore=["lm_head"])
    if algo == "gptq":
        return GPTQModifier(config_groups=eb.w_scheme(bits, 128, symmetric=True), ignore=["lm_head"],
                            actorder=None, dampening_frac=0.01)
    return AWQModifier(config_groups=eb.w_scheme(bits, 128, symmetric=False), ignore=["lm_head"])

def run(cfg_id, algo, bits, calib, post):
    awq = algo == "awq"
    dtype = torch.float32 if awq else torch.float16           # AWQ's scale search gives NaN in fp16 on a T4
    # AWQ: load on CPU so the pipeline moves one layer at a time to the GPU (keeps the T4 from running out of memory)
    model = AutoModelForCausalLM.from_pretrained(MODEL_ID, dtype=dtype, device_map=None if awq else "cuda")
    torch.cuda.reset_peak_memory_stats()
    with eb.Timer() as t:
        if calib is None:
            oneshot(model=model, recipe=make_recipe(algo, bits))
        else:
            oneshot(model=model, dataset=CALIB[calib], recipe=make_recipe(algo, bits),
                    max_seq_length=eb.CFG["calib_len"], num_calibration_samples=eb.CFG["calib_n"])
    qpeak = eb.peak_vram_gb()
    model.to(torch.float16)                                    # every row is evaluated in fp16
    dispatch_for_generation(model)
    res = eb.evaluate(model, tok, ref=ref)
    out = eb.models_dir() / cfg_id
    model.save_pretrained(out, save_compressed=True); tok.save_pretrained(out)
    size = eb.dir_size_gb(out)
    eb.log_result(cfg_id, post=post, method=algo.upper(), bits=bits, group=128, calib=calib,
                  calib_n=eb.CFG["calib_n"] if calib else None, calib_dtype=str(dtype).replace("torch.", ""),
                  quant_time_s=t.s, quant_peak_vram_gb=qpeak, size_gb=size, bpw=eb.bpw(size), **res)
    del model; eb.reset_llmc(); eb.free_gpu(); print(eb.ram_gb())

for c in CONFIGS:
    if c[0] in done_ids():
        print("skip", c[0]); continue
    run(*c)

## NF4 (bitsandbytes, double quant, no calibration)

In [ ]:
if "nf4_dq" not in done_ids():
    bnb = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_quant_type="nf4", bnb_4bit_use_double_quant=True,
                             bnb_4bit_compute_dtype=torch.float16)
    with eb.Timer() as t:
        model = AutoModelForCausalLM.from_pretrained(MODEL_ID, quantization_config=bnb, device_map="cuda",
                                                     dtype=torch.float16)
    res = eb.evaluate(model, tok, ref=ref)
    out = eb.models_dir() / "nf4_dq"
    model.save_pretrained(out); tok.save_pretrained(out)
    size = eb.dir_size_gb(out)
    eb.log_result("nf4_dq", post=5, method="NF4", bits=4, group=64, double_quant=True, quant_time_s=t.s,
                  size_gb=size, bpw=eb.bpw(size), **res)
    del model; eb.free_gpu()

## T4 speed (vLLM in its own venv; first call installs it, ~3–5 min)
Runs one model at a time with nothing else on the GPU. If the session restarted, the 4-bit checkpoints on local disk
are gone: the cell re-makes them (speed doesn't depend on calibration, so AWQ uses a short 512-token set here).
3-bit has no fast vLLM kernel, and recent vLLM no longer serves bitsandbytes NF4, so those rows are accuracy-only.

In [ ]:
del ref; eb.free_gpu()

def ensure_checkpoint(cfg_id, algo):
    out = eb.models_dir() / cfg_id
    if (out / "config.json").exists(): return out
    awq = algo == "awq"; seqlen = 512 if awq else eb.CFG["calib_len"]
    m = AutoModelForCausalLM.from_pretrained(MODEL_ID, dtype=torch.float32 if awq else torch.float16,
                                             device_map=None if awq else "cuda")
    oneshot(model=m, dataset=eb.calib_dataset(tok, "c4", seqlen=seqlen), recipe=make_recipe(algo, 4),
            max_seq_length=seqlen, num_calibration_samples=eb.CFG["calib_n"])
    m.to(torch.float16); m.save_pretrained(out, save_compressed=True); tok.save_pretrained(out)
    del m; eb.reset_llmc(); eb.free_gpu()
    return out

def has_speed(cfg_id):
    p = rd / RESULTS_FILE
    return any(json.loads(l).get("config_id") == cfg_id and f"{eb.gpu_tag()}_decode_tok_s" in l
               for l in open(p) if l.strip()) if p.exists() else False

for cfg_id, algo in [("fp16", None), ("gptq_w4g128", "gptq"), ("awq_w4g128", "awq")]:
    if has_speed(cfg_id): print("skip", cfg_id); continue
    path = MODEL_ID if algo is None else ensure_checkpoint(cfg_id, algo)
    spd = eb.vllm_speed(path)
    print(cfg_id, spd)
    if spd: eb.log_result(cfg_id, engine="vLLM", **eb.prefixed(spd, eb.gpu_tag()))

## Copy the results
Downloads from Kaggle can fail: the cell below prints the file so you can copy it.

In [ ]:
print(open(rd / "results_01_ptq.jsonl").read())